This notebook (will) import~~s~~ & perform basic analysis of the shendure data using our package.

In [2]:
#general imports
import sys
import os
import statsmodels.discrete.count_model as smdc
import patsy
from tensorzinb.tensorzinb import TensorZINB
from formulaic import Formula
import pandas as pd
import numpy as np

data_root="/gpfs/gibbs/pi/reilly/tabula_data"

2025-04-16 16:17:05.586352: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-16 16:17:05.623506: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
#dask imports
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
from dask import delayed

import socket

In [4]:
sh_dat=pd.read_csv(f"{data_root}/shendure/shendure_counts_grouped.txt",sep="\t")

In [5]:
modelspecs=pd.read_csv("modelspecs.tsv",sep="\t",index_col=0)
modelspecs

,Dataset,Hardware,DNA,Main Equ Type,Z equ type,Equation main,Equation Z,Broken_by
Code,,,,,,,,
c00000,Shendure (0),Statsmodels (0),No dna (0),simple addition (0),replicate (0),umis_mpra_bc ~ C(cre_id) + C(cell_type) -1,C(rep_id),NaN
c01000,Shendure (0),Tensorzinb a100 (1),No dna (0),simple addition (0),replicate (0),umis_mpra_bc ~ C(cre_id) + C(cell_type) -1,C(rep_id),NaN


In [14]:
@delayed
def create_matricies(main_form,zin_form,data):
    y, X=Formula(main_form).get_model_matrix(data,output='pandas')
    Z=Formula(zin_form).get_model_matrix(data,output='pandas')
    return(X, y, Z)


#print(create_matricies(main_form=modelspecs.iloc[0]["Equation main"],zin_form=modelspecs.iloc[0]["Equation Z"],data=sh_dat))

def create_cluster():
    """
    Makes a simple slurm cluster with some preset parameters.
    """

    cluster=SLURMCluster(
        cores=2,#cores per slurm job
        memory="16G",#memory per slurm job
        processes=1,#dask workers per slurm job
        job_extra=["-p ycga", f"--job-name=simclust","--time=08:00:00"]
    )

    client = Client(cluster)

    print(f"Cluster started. Monitor on {cluster.dashboard_link}")

    cluster.scale(jobs=1)
    #cluster.adapt(minimum_jobs=1, maximum_jobs=10)

    return cluster,client

def kill_cluster(client):
    client.shutdown()

#we will start up the cluster, run, kill for each, then get job statistics from sacct
#each cluster will get an ID & put it in the name of the job & save that ID for later 
#sacct summary.. 
#collect with subprocess query

@delayed
def statsmodels_fit(X,y,Z):
    zinb_model = smdc.ZeroInflatedNegativeBinomialP(y, X, exog_infl=Z)

    n_count_params = zinb_model.exog.shape[1]      # Count model parameters
    n_infl_params = zinb_model.exog_infl.shape[1]    # Inflation model parameters
    n_total = n_count_params + n_infl_params + 1 # adding 1 for alpha
    start_params = np.full(n_total, 0.1)

    zinb_result = zinb_model.fit(start_params=start_params,maxiter=1000,method="cg")

    return zinb_result

def bench(row):
    print(f"[+] Fitting {modelspecs.iloc[0].name}",flush=True)
    print(f"[+] Creating cluster")
    #if "Statsmodels" in modelspecs.iloc[row]['Hardware']:
    #    print("[+] No GPUs needed.")
    cluster,client=create_cluster()

    print("[+] Scattering data")
    sh_dat_fut = client.scatter(sh_dat, broadcast=True)

    print("[+] Creating matricies")
    mats = create_matricies(
        main_form=modelspecs.iloc[row]["Equation main"],
        zin_form=modelspecs.iloc[row]["Equation Z"],
        data=sh_dat_fut
    )

    X, y, Z=mats.compute()

    print("[+] Matricies done. Fitting model")

    model=None

    #little rats-nest to handle a couple possibilities...
    if pd.isnull(modelspecs.iloc[row]['Broken_by']):
        #model not parallelizably 

        if "Statsmodels" in modelspecs.iloc[row]['Hardware']:
            print("[+] Creating statsmodels model")
            model=statsmodels_fit(X,y,Z)

    print("[+] Fitting...")

    model=model.compute()

    print("[+] Done!")

    print(f"job_ids: {cluster.job_ids}")

    kill_cluster(client)
    

    return model

    


In [15]:
model=bench(row=0)

[+] Fitting c00000
[+] Creating cluster
Cluster started. Monitor on http://10.178.138.14:8787/status
[+] Scattering data
[+] Creating matricies
[+] Matricies done. Fitting model
[+] Creating statsmodels model
[+] Fitting...


In [ ]:
print(model)
import pickle
import datetime

# 1. Print the current time
now = datetime.datetime.now()
print(f"Current time: {now.strftime('%Y-%m-%d %H:%M:%S')}")



# 3. Pickle and dump the object to disk
with open('c00000.pkl', 'wb') as f:
    pickle.dump(model, f)

print()

In [ ]:
#print("[+] Dumping model to disc.")

#print("[+] Collecting statistics...")

#print("[+] Dumping statistics")